# Импорты

In [1]:
import plotly.express as px
import pandas as pd
import numpy as np

# Входные данные

Уравнение:<br>
$\displaystyle
u'_t(x, t) = \frac{16}{\pi} u''_{xx}(x, t) + tx(1 - x) \\
0 < x < 4 \\
0 < t < T
$

Начальные и краевые условия:<br>
$\displaystyle
u(x, 0) = x(x^2 - x), ~~ x \in [0; 4] \\
u(0, t) = 0 \\
u(4, t) = 0
$

Сетка:<br>
$\displaystyle
\{x_m, t_n\}, ~~ m = \overline{0, M}, ~ n = \overline{0, N} \\
x_m = m h \\
h = \frac{4}{M} \\
t_n = n \tau \\
\tau = \frac{T}{N} \\
$

In [ ]:
a2 = 16 / np.pi

def phi(x: float, t: float) -> float:
    return t * x * (1 - x)

def psi(x: float) -> float:
    return x * (x ** 2 - x)

def gamma_0(t: float) -> float:
    return 0

def gamma_1(t: float) -> float:
    return 0

l = 4
T = 0.1

M = 1000
N = 600

h = l / M
tau = T / N

x = np.linspace(0, l, M + 1)
t = np.linspace(0, T, N + 1)

# Решение задачи

## Явная схема

Схема:<br>
$\displaystyle
\frac{u_j^{k + 1} - u_j^k}{\tau} = \frac{16}{\pi} \cdot \frac{u_{j + 1}^k - 2u_j^k + u_{j - 1}^k}{h^2} + t_k x_j (1 - x_j)
$

Начальные и граничные условия:<br>
$\displaystyle
u_j^0 = x_j (x_j^2 - x_j) \\
u_0^k = 0 \\
u_M^k = 0
$

Система:<br>
$\displaystyle
u_j^{k + 1} = \frac{16 \tau}{\pi} \cdot \frac{u_{j + 1}^k - 2u_j^k + u_{j - 1}^k}{h^2} + u_j^k + \tau t_k x_j (1 - x_j), ~~ k = \overline{0, N - 1}
$

Условие сходимости:<br>
$\displaystyle
\frac{a^2 \tau}{h^2} = \frac{16 \tau}{\pi h^2} \leq \frac{1}{2} \Rightarrow \tau \leq \frac{\pi h^2}{32} \\
\frac{T}{N} \leq \frac{\pi}{2 M^2} \Rightarrow M \in \left( 0; \sqrt{ \frac{N \pi}{2T}} \right]
$

In [3]:
u_h_tau__explicit = np.zeros((M + 1, N + 1))

u_h_tau__explicit[:, 0] = psi(x)
u_h_tau__explicit[0, :] = gamma_0(t)
u_h_tau__explicit[M, :] = gamma_1(t)

for k in range(0, N):
    for j in range(1, M):
        u_h_tau__explicit[j, k + 1] = 16 * tau / np.pi * (u_h_tau__explicit[j + 1, k] - 2 * u_h_tau__explicit[j, k] + u_h_tau__explicit[j - 1, k]) + u_h_tau__explicit[j, k] + tau * phi(x[j], t[k])

print("Выполнение устойчивости:", tau <= (np.pi * h ** 2 / 32))

Выполнение устойчивости: False


## Неявная схема

Схема:<br>
$\displaystyle
\frac{u_j^{k + 1} - u_j^k}{\tau} = \frac{16}{\pi} \cdot \frac{u_{j + 1}^{k + 1} - 2 u_j^{k + 1} + u_{j - 1}^{k + 1}}{h^2} + t_{k + 1} x_j (1 - x_j)
$

Начальные и граничные условия:<br>
$\displaystyle
u_j^0 = x_j (x_j^2 - x_j) \\
u_0^k = 0 \\
u_M^k = 0
$

Система:<br>
$\displaystyle
-\frac{16 \tau}{\pi h^2} u_{j + 1}^{k + 1} + \left( \frac{32 \tau}{\pi h^2} + 1 \right) u_j^{k + 1} -\frac{16 \tau}{\pi h^2} u_{j - 1}^{k + 1} = u_j^k + \tau t_{k + 1} x_j (1 - x_j), ~~ k = \overline{0, N - 1}
$

$
\forall k = \overline{0, N - 1} \\
\left( \begin{array}{ccccccccc}
    \displaystyle \frac{32 \tau}{\pi h^2} + 1 & \displaystyle -\frac{16 \tau}{\pi h^2} & 0 & 0 & \ldots & 0 & 0 & 0 & 0 \\
    \displaystyle -\frac{16 \tau}{\pi h^2} & \displaystyle \frac{32 \tau}{\pi h^2} + 1 & \displaystyle -\frac{16 \tau}{\pi h^2} & 0 & \ldots & 0 & 0 & 0 & 0 \\
    0 & \displaystyle -\frac{16 \tau}{\pi h^2} & \displaystyle \frac{32 \tau}{\pi h^2} + 1 & \displaystyle -\frac{16 \tau}{\pi h^2} & \ldots & 0 & 0 & 0 & 0 \\
    \vdots & \vdots & \vdots & \vdots & \ddots & \vdots & \vdots & \vdots & \vdots \\
    0 & 0 & 0 & 0 & \ldots & \displaystyle -\frac{16 \tau}{\pi h^2} & \displaystyle \frac{32 \tau}{\pi h^2} + 1 & \displaystyle -\frac{16 \tau}{\pi h^2} & 0 \\
    0 & 0 & 0 & 0 & \ldots & 0 & \displaystyle -\frac{16 \tau}{\pi h^2} & \displaystyle \frac{32 \tau}{\pi h^2} + 1 & \displaystyle -\frac{16 \tau}{\pi h^2} \\
    0 & 0 & 0 & 0 & \ldots & 0 & 0 & \displaystyle -\frac{16 \tau}{\pi h^2} & \displaystyle \frac{32 \tau}{\pi h^2} + 1
\end{array} \right) \cdot \left( \begin{array}{c}
    u_1^{k + 1} \\ u_2^{k + 1} \\ u_3^{k + 1} \\ \vdots \\ u_{M - 3}^{k + 1} \\ u_{M - 2}^{k + 1} \\ u_{M - 1}^{k + 1}
\end{array} \right) = \left( \begin{array}{c}
    u_1^k + \tau t_{k + 1} x_1 (1 - x_1) \\ u_2^k + \tau t_{k + 1} x_2 (1 - x_2) \\ u_3^k + \tau t_{k + 1} x_3 (1 - x_3) \\ \vdots \\ u_{M - 3}^k + \tau t_{k + 1} x_{M - 3} (1 - x_{M - 3}) \\ u_{M - 2}^k + \tau t_{k + 1} x_{M - 2} (1 - x_{M - 2}) \\ u_{M - 1}^k + \tau t_{k + 1} x_{M - 1} (1 - x_{M - 1})
\end{array} \right)
$

In [4]:
u_h_tau__implicit = np.zeros((M + 1, N + 1))

u_h_tau__implicit[:, 0] = psi(x)
u_h_tau__implicit[0, :] = gamma_0(t)
u_h_tau__implicit[M, :] = gamma_1(t)

def F(j:int, k: int) -> float:
    return u_h_tau__implicit[j, k] + tau * phi(x[j], t[k + 1])

A = np.full(M - 1, -16 * tau / (np.pi * h ** 2))
B = np.full(M - 1, 32 * tau / (np.pi * h ** 2) + 1)
C = np.full(M - 1, -16 * tau / (np.pi * h ** 2))

A[0] = 0
C[M - 2] = 0

for k in range(0, N):
    alpha = np.zeros(M - 1)
    beta = np.zeros(M - 1)

    alpha[1] = -C[0] / B[0]
    beta[1] = F(1, k) / B[0]

    for j in range(1, M - 2):
        alpha[j + 1] = -C[j] / (B[j] + alpha[j] * A[j])
        beta[j + 1] = (F(j + 1, k) - beta[j] * A[j]) / (B[j] + alpha[j] * A[j])

    u_h_tau__implicit[M - 1, k + 1] = (F(M - 1, k) - beta[M - 2] * A[M - 2]) / (B[M - 2] + alpha[M - 2] * A[M - 2])

    for j in range(M - 2, 0, -1):
        u_h_tau__implicit[j, k + 1] = alpha[j] * u_h_tau__implicit[j + 1, k + 1] + beta[j]

## Схема Кранка-Николсон

Схема:<br>
$\displaystyle
\frac{u_j^{k + 1} - u_j^k}{\tau} = \theta \left( \frac{16}{\pi} \cdot \frac{u_{j + 1}^{k + 1} - 2u_j^{k + 1} + u_{j - 1}^{k + 1}}{h^2} + t_{k + 1} x_j (1 - x_j) \right) + (1 - \theta) \left( \frac{16}{\pi} \cdot \frac{u_{j + 1}^k - 2u_j^k + u_{j - 1}^k}{h^2} + t_k x_j (1 - x_j) \right) \\
$

Начальные и граничные условия:<br>
$\displaystyle
u_j^0 = x_j (x_j^2 - x_j) \\
u_0^k = 0 \\
u_M^k = 0
$

Система:<br>
$\displaystyle
\sqsupset \theta = \frac{1}{2} \\
-\frac{8 \tau}{\pi h^2} u_{j - 1}^{k + 1} + \left( \frac{16 \tau}{\pi h^2} + 1 \right) u_j^{k + 1} - \frac{8 \tau}{\pi h^2} u_{j + 1}^{k + 1} = \frac{8 \tau}{\pi h^2} u_{j - 1}^k + \left( 1 - \frac{16 \tau}{\pi h^2} \right) u_j^k + \frac{8 \tau}{\pi h^2} u_{j + 1}^k + \frac{\tau}{2} x_j (1 - x_j)(t_{k + 1} + t_k), ~~ k = \overline{0, N - 1}
$

$
\forall k = \overline{0, N - 1} \\
\left( \begin{array}{ccccccccc}
    \displaystyle \frac{16 \tau}{\pi h^2} + 1 & \displaystyle -\frac{8 \tau}{\pi h^2} & 0 & 0 & \ldots & 0 & 0 & 0 & 0 \\
    \displaystyle -\frac{8 \tau}{\pi h^2} & \displaystyle \frac{16 \tau}{\pi h^2} + 1 & \displaystyle -\frac{8 \tau}{\pi h^2} & 0 & \ldots & 0 & 0 & 0 & 0 \\
    0 & \displaystyle -\frac{8 \tau}{\pi h^2} & \displaystyle \frac{16 \tau}{\pi h^2} + 1 & \displaystyle -\frac{8 \tau}{\pi h^2} & \ldots & 0 & 0 & 0 & 0 \\
    \vdots & \vdots & \vdots & \vdots & \ddots & \vdots & \vdots & \vdots & \vdots \\
    0 & 0 & 0 & 0 & \ldots & \displaystyle -\frac{8 \tau}{\pi h^2} & \displaystyle \frac{16 \tau}{\pi h^2} + 1 & \displaystyle -\frac{8 \tau}{\pi h^2} & 0 \\
    0 & 0 & 0 & 0 & \ldots & 0 & \displaystyle -\frac{8 \tau}{\pi h^2} & \displaystyle \frac{16 \tau}{\pi h^2} + 1 & \displaystyle -\frac{8 \tau}{\pi h^2} \\
    0 & 0 & 0 & 0 & \ldots & 0 & 0 & \displaystyle -\frac{8 \tau}{\pi h^2} & \displaystyle \frac{16 \tau}{\pi h^2} + 1
\end{array} \right) \cdot \left( \begin{array}{c}
    u_1^{k + 1} \\ u_2^{k + 1} \\ u_3^{k + 1} \\ \vdots \\ u_{M - 3}^{k + 1} \\ u_{M - 2}^{k + 1} \\ u_{M - 1}^{k + 1}
\end{array} \right) = \left( \begin{array}{c}
    \displaystyle \frac{8 \tau}{\pi h^2} u_0^k + \left( 1 - \frac{16 \tau}{\pi h^2} \right) u_1^k + \frac{8 \tau}{\pi h^2} u_2^k + \frac{\tau}{2} x_1 (1 - x_1)(t_{k + 1} + t_k) \\ 
    \displaystyle \frac{8 \tau}{\pi h^2} u_1^k + \left( 1 - \frac{16 \tau}{\pi h^2} \right) u_2^k + \frac{8 \tau}{\pi h^2} u_3^k + \frac{\tau}{2} x_2 (1 - x_2)(t_{k + 1} + t_k) \\ 
    \displaystyle \frac{8 \tau}{\pi h^2} u_2^k + \left( 1 - \frac{16 \tau}{\pi h^2} \right) u_3^k + \frac{8 \tau}{\pi h^2} u_4^k + \frac{\tau}{2} x_3 (1 - x_3)(t_{k + 1} + t_k) \\ 
    \vdots \\ 
    \displaystyle \frac{8 \tau}{\pi h^2} u_{M - 4}^k + \left( 1 - \frac{16 \tau}{\pi h^2} \right) u_{M - 3}^k + \frac{8 \tau}{\pi h^2} u_{M - 2}^k + \frac{\tau}{2} x_{M - 3} (1 - x_{M - 3})(t_{k + 1} + t_k) \\
    \displaystyle \frac{8 \tau}{\pi h^2} u_{M - 3}^k + \left( 1 - \frac{16 \tau}{\pi h^2} \right) u_{M - 2}^k + \frac{8 \tau}{\pi h^2} u_{M - 1}^k + \frac{\tau}{2} x_{M - 2} (1 - x_{M - 2})(t_{k + 1} + t_k) \\
    \displaystyle \frac{8 \tau}{\pi h^2} u_{M - 2}^k + \left( 1 - \frac{16 \tau}{\pi h^2} \right) u_{M - 1}^k + \frac{8 \tau}{\pi h^2} u_M^k + \frac{\tau}{2} x_{M - 1} (1 - x_{M - 1})(t_{k + 1} + t_k) \\
\end{array} \right)
$

In [5]:
u_h_tau__crank_nicolson = np.zeros((M + 1, N + 1))

u_h_tau__crank_nicolson[:, 0] = psi(x)
u_h_tau__crank_nicolson[0, :] = gamma_0(t)
u_h_tau__crank_nicolson[M, :] = gamma_1(t)

def F(j:int, k: int) -> float:
    return  8 * tau / (np.pi * h ** 2) * u_h_tau__crank_nicolson[j - 1, k] + \
            (1 - 16 * tau / (np.pi * h ** 2)) * u_h_tau__crank_nicolson[j, k] + \
            8 * tau / (np.pi * h ** 2) * u_h_tau__crank_nicolson[j + 1, k] + \
            tau / 2 * phi(x[j], t[k + 1]) + tau / 2 * phi(x[j], t[k])

A = np.full(M - 1, -8 * tau / (np.pi * h ** 2))
B = np.full(M - 1, 16 * tau / (np.pi * h ** 2) + 1)
C = np.full(M - 1, -8 * tau / (np.pi * h ** 2))

A[0] = 0
C[M - 2] = 0

for k in range(0, N):
    alpha = np.zeros(M - 1)
    beta = np.zeros(M - 1)

    alpha[1] = -C[0] / B[0]
    beta[1] = F(1, k) / B[0]

    for j in range(1, M - 2):
        alpha[j + 1] = -C[j] / (B[j] + alpha[j] * A[j])
        beta[j + 1] = (F(j + 1, k) - beta[j] * A[j]) / (B[j] + alpha[j] * A[j])

    u_h_tau__crank_nicolson[M - 1, k + 1] = (F(M - 1, k) - beta[M - 2] * A[M - 2]) / (B[M - 2] + alpha[M - 2] * A[M - 2])

    for j in range(M - 2, 0, -1):
        u_h_tau__crank_nicolson[j, k + 1] = alpha[j] * u_h_tau__crank_nicolson[j + 1, k + 1] + beta[j]

# Сравнение

In [6]:
def prep_df(u: np.ndarray, name: str, x: np.ndarray, t: np.ndarray) -> pd.DataFrame:
    x_j, t_k = np.meshgrid(x, t, indexing='ij')
    return pd.DataFrame({
        'x': x_j.flatten(),
        't': t_k.flatten(),
        'u': u.flatten(),
        'Схема': name
    })

df = pd.concat([
    prep_df(u_h_tau__explicit, "Явная", x, t),
    prep_df(u_h_tau__implicit, "Неявная", x, t),
    prep_df(u_h_tau__crank_nicolson, "Кранка-Николсон", x, t)
])
df

,x,t,u,Схема
0,0.0,0.000000,0.0,Явная
1,0.0,0.000167,0.0,Явная
2,0.0,0.000333,0.0,Явная
3,0.0,0.000500,0.0,Явная
4,0.0,0.000667,0.0,Явная
...,...,...,...,...
601596,4.0,0.099333,0.0,Кранка-Николсон
601597,4.0,0.099500,0.0,Кранка-Николсон
601598,4.0,0.099667,0.0,Кранка-Николсон
601599,4.0,0.099833,0.0,Кранка-Николсон


## Срез по $t$

In [7]:
fig = px.line(
    df, x="x", y="u", color="Схема", 
    animation_frame="t",
    title="Сравнение профилей решения в моменты времени t"
)

fig.update_layout(xaxis_title="Координата X", yaxis_title="Значение U")

fig.show()